**# Step 0: data preprocessing #**

**Substep 0.1** First of all we need to download publicly available data. In our case, we download the SRR files of the project through a slurm script. 

Creating the download_srr.sbatch script to download fastq.gz according to the list of SRA IDs

In [ ]:
#!/bin/bash                                                                       # shebang: the script is executed in bash
#SBATCH -J download_srr                                                           # -J: job name, will be visible in the job queue
#SBATCH -n 1                                                                      # -n: number of CPU cores/tasks. Here 1 is a single-threaded script
#SBATCH --mem=8G                                                                  # RAM request: 8 GB for the entire task
#SBATCH -t 24:00:00                                                               # -t: the maximum working time (walltime) is 24 hours, if it is not completed during this time, the process will crash
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/download_srr_%j.err          # --error: where to write stderr (errors). %j is automatically replaced by the job ID.
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/download_srr_%j.out         # --output: where to write stdout (normal program output)

cd /mnt/tank/scratch/ris/SRR_files                                                # go to the working directory, where the downloaded SRR files and the source script itself will be located  
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh                            # loading the initialization script mamba
mamba activate snakemake                                                          # activating the environment (environment name "snakemake")

# Download (example for a single file, it is better to use --split-files for paired ends)
fasterq-dump --split-files --outdir . SRR25006867
gzip *.fastq

# Note: The SRR list loop was used for bulk upload

**Substep 0.2** Primary Quality Control (FastQC) on raw data

Creating the run_fastqc_and_parse.sbatch script, which runs FastQC on all fastqs.gz and then parses the reports.

The file parse_fastqc.py collects the total number of reads (the sum of R1+R2) and the length of reads from FastQC reports for each sample, storing them in fastqc_summary.csv.

In [ ]:
#!/bin/bash
#SBATCH -J fastqc_parse
#SBATCH -n 1
#SBATCH --cpus-per-task=2
#SBATCH --mem=16G
#SBATCH -t 24:00:00
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/fastqc_parse_%j.err
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/fastqc_parse_%j.out

cd /mnt/tank/scratch/ris/SRR_files
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

fastqc --threads 2 *.fastq.gz
python parse_fastqc.py

parse_fastqc.py

In [ ]:
#!/usr/bin/env python
import os
import glob
import pandas as pd
import zipfile
import io

fastq_files = glob.glob("*.fastq.gz")
sample_set = set()
for f in fastq_files:
    if f.endswith("_1.fastq.gz"):
        sample_set.add(f[:-11])
    elif f.endswith("_2.fastq.gz"):
        sample_set.add(f[:-11])

samples = sorted(sample_set)
print(f"Found {len(samples)} samples: {samples}")

results = []
missing = []

for sample in samples:
    r1_zip = f"{sample}_1_fastqc.zip"
    r2_zip = f"{sample}_2_fastqc.zip"

    if not os.path.isfile(r1_zip):
        missing.append(r1_zip)
        continue
    if not os.path.isfile(r2_zip):
        missing.append(r2_zip)
        continue

    reads = 0
    lengths = []

    def read_fastqc_data(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            base = os.path.basename(zip_path).replace('.zip', '')
            data_file = f"{base}/fastqc_data.txt"
            with z.open(data_file) as f:
                return io.TextIOWrapper(f).readlines()

    lines = read_fastqc_data(r1_zip)
    for line in lines:
        if "Total Sequences" in line:
            reads += int(line.split()[-1])
        if "Sequence length" in line:
            lengths.append(line.split()[-1].strip())

    lines = read_fastqc_data(r2_zip)
    for line in lines:
        if "Total Sequences" in line:
            reads += int(line.split()[-1])
        if "Sequence length" in line:
            lengths.append(line.split()[-1].strip())

    results.append({
        "sample": sample,
        "total_reads": reads,
        "read1_length": lengths[0] if len(lengths) > 0 else "NA",
        "read2_length": lengths[1] if len(lengths) > 1 else "NA"
    })

    print(sample, reads, lengths)

if results:
    pd.DataFrame(results).to_csv("fastqc_summary.csv", index=False)
    print("\nSummary saved to fastqc_summary.csv")
else:
    print("No results collected.")

if missing:
    print("\nWarning: missing FastQC zip files:")
    for m in missing:
        print("  ", m)

**Substep 0.3** Trimming (cleaning) with the help of a Trimomatic

The run_trimmomatic.sbatch script processes all paired files.

In [ ]:
#!/bin/bash
#SBATCH -J trimmomatic
#SBATCH -n 1
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH -t 06:00:00
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/trimmomatic_%j_%a.err
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/trimmomatic_%j_%a.out

cd /mnt/tank/scratch/ris/SRR_files
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

samples=($(ls *_1.fastq.gz | sed 's/_1.fastq.gz//'))
sample=${samples[$SLURM_ARRAY_TASK_ID-1]}

trimmomatic PE -threads 2 \
  ${sample}_1.fastq.gz ${sample}_2.fastq.gz \
  ${sample}_1_trimmed_paired.fastq.gz ${sample}_1_trimmed_unpaired.fastq.gz \
  ${sample}_2_trimmed_paired.fastq.gz ${sample}_2_trimmed_unpaired.fastq.gz \
  ILLUMINACLIP:/path/to/adapters.fa:2:30:10 LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
  
# After trimming, files with the suffix _trimmed_paired appear.fastq.gz.

**Substep 0.4** FastQC on clean data

The run_fastqc_clean.sbatch script runs FastQC on *_trimmed_paired.fastq.gz and then parses them with parse_fastqc_clean.py (similar to clause 1.2, but taking into account suffixes).

In [ ]:
#!/bin/bash
#SBATCH -J fastqc_clean
#SBATCH -n 1
#SBATCH --cpus-per-task=2
#SBATCH --mem=16G
#SBATCH -t 24:00:00
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/fastqc_clean_%j.err

cd /mnt/tank/scratch/ris/SRR_files
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

fastqc --threads 2 *_trimmed_paired.fastq.gz
python parse_fastqc_clean.py

# Output file: fastqc_clean_summary.csv

parse_fastqc_clean.py

In [ ]:
import os
import zipfile
import io
import glob
import pandas as pd

def read_fastqc_data(zip_path):
    """
    Extracts the contents fastqc_data.txt from the FastQC zip archive.
    Returns a list of rows.
    """
    with zipfile.ZipFile(zip_path, 'r') as z:
        # File name inside the archive: <archive_name without_zip>/fastqc_data.txt
        base = os.path.basename(zip_path).replace('.zip', '')
        data_file = f"{base}/fastqc_data.txt"
        with z.open(data_file) as f:
            return io.TextIOWrapper(f).readlines() # wraps a binary stream into a text stream (using UTF‑8 encoding) to read strings

def main():
   # Finding all R1 files for pure data - suffix _1_trimmed_paired.fastq.gz
    r1_files = glob.glob("*_1_trimmed_paired.fastq.gz")
    if not r1_files:
        print("No files with the suffix '_1_trimmed_paired were found.fastq.gz'")
        return

    # Extract the names of the samples-remove the suffix
    samples = set()
    for f in r1_files:
        # Expected format: <sample>_1_trimmed_paired.fastq.gz
        sample = f.replace('_1_trimmed_paired.fastq.gz', '')
        samples.add(sample)
    samples = sorted(samples)
    print(f"Found {len(samples)} samples: {samples}")

    results = []
    missing = []

    for sample in samples:
        # Forming the expected names of FastQC zip archives
        r1_zip = f"{sample}_1_trimmed_paired_fastqc.zip"
        r2_zip = f"{sample}_2_trimmed_paired_fastqc.zip"

        # Check for both zip files
        if not os.path.isfile(r1_zip):
            missing.append(r1_zip)
            continue
        if not os.path.isfile(r2_zip):
            missing.append(r2_zip)
            continue

        total_reads = 0
        lengths = []

        # R1 parsing
        lines = read_fastqc_data(r1_zip)
        for line in lines:
            if "Total Sequences" in line:
                total_reads += int(line.split()[-1])
            if "Sequence length" in line:
                lengths.append(line.split()[-1].strip())

        # R2 parsing
        lines = read_fastqc_data(r2_zip)
        for line in lines:
            if "Total Sequences" in line:
                total_reads += int(line.split()[-1])
            if "Sequence length" in line:
                lengths.append(line.split()[-1].strip())

        results.append({
            "sample": sample,
            "total_reads": total_reads,
            "read1_length": lengths[0] if len(lengths) > 0 else "NA",
            "read2_length": lengths[1] if len(lengths) > 1 else "NA"
        })

        print(sample, total_reads, lengths)

    # Results saving
    if results:
        df = pd.DataFrame(results)
        df.to_csv("fastqc_clean_summary.csv", index=False)
        print("\nThe result is saved in fastqc_clean_summary.csv")
    else:
        print("There is no data collected")

    if missing:
        print("\nThe following FastQC zip files are missing:")
        for m in missing:
            print("  ", m)

if __name__ == "__main__":
    main()

**Substep 0.5** Comparison of the number of reeds before and after trimming

The script is running on the local computer (or on the server) compare_reads.py , which builds a boxplot and counts statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

raw = pd.read_csv('fastqc_summary.csv')
clean = pd.read_csv('fastqc_clean_summary.csv')
raw = raw[['sample', 'total_reads']].rename(columns={'total_reads': 'raw_reads'})
clean = clean[['sample', 'total_reads']].rename(columns={'total_reads': 'clean_reads'})
df = pd.merge(raw, clean, on='sample')

df_melt = df.melt(id_vars='sample')
sns.boxplot(x='variable', y='value', data=df_melt)
plt.savefig('reads_boxplot.png')

df['retained'] = df['clean_reads'] / df['raw_reads']
print(df['retained'].describe())
stat, p = ttest_rel(df['raw_reads'], df['clean_reads'])
print(f'p-value: {p}')

# In our case, the result is a boxplot graph with a p-value of ~1.56e-34 (the difference is significant, but the loss is < 0.1%)

**Substep 0.6** Based on the metadata attached to the metagenomes, we recalculate the values of BMD (bone density, femoral neck) into T_score values, taking into account the reference values for Europoid women

In [ ]:
# BMD
import pandas as pd
import numpy as np
metadata = pd.read_csv('./SraRunTable.csv')
metadata.rename(columns={'Run': 'sample'}, inplace=True)
metadata = metadata[['sample','Fracture', 'HTOT_BMD_(g/cm2)', 'age', 'BMI']]

df = metadata.copy()

df.rename(columns={'HTOT_BMD_(g/cm2)': 'BMD_hip'}, inplace=True)

# References for Europoid women (60+, total hip)
REF_MEAN_HIP = 0.892   # g/cm²
REF_SD_HIP = 0.120

# T-score calculation
df['T_score'] = (df['BMD_hip'] - REF_MEAN_HIP) / REF_SD_HIP

# 0 = normal (T ≥ -1), 1 = osteopinia(osteoporosis) (T < -1)
df['BMD_group'] = (df['T_score'] < -1.0).astype(int)

# Categories
df['BMD_category'] = pd.cut(df['T_score'], 
                            bins=[-np.inf, -2.5, -1.0, np.inf],
                            labels=['Osteoporosis', 'Osteopenia', 'Normal'])

**Initial data**

*Folder with reads*: /mnt/tank/scratch/rislamova/SRR_files/

*Files*: SRR*_1_trimmed_paired.fastq.gz and SRR*_2_trimmed_paired.fastq.gz (compressed)

*Metadata*: SraRunTable.csv with columns Run (sample ID) and Fracture (0 – healthy, >=1 – sick)